# Vectur
## Online Pen Trajectory Generation for Urdu Handwriting

- Danish Munib (461050)
- Rafay Ahmad (473383)
- BESE-14 B

### Setup

In [10]:
ROOT_DIR = '/mnt/vectur/'

In [ ]:
%%bash
apt-get update
apt-get install unrar-free tree

In [ ]:
%uv pip install pyarrow scipy opencv-python networkx Pillow arabic-reshaper python-bidi scikit-image

In [ ]:
%%bash
unrar-free x /mnt/vectur/handwriting_dataset.rar /mnt/vectur/
mv /mnt/vectur/data_online_line_width_alpha/ /mnt/vectur/hwd/

#### Colab

In [ ]:
!apt-get install unrar tree

In [ ]:
!uv pip install pyarrow alive-progress

In [ ]:
!unrar x /content/drive/MyDrive/TUKL/data_online_line_width_alpha.rar "*.csv" /content/
!mv /content/data_online_line_width_alpha/ /content/hwd/
!tree -L 1 /content/hwd/

### Data Prep

#### CSV Handling

In [ ]:
import pandas as pd
import glob

main_files = glob.glob(ROOT_DIR + 'hwd/main_*.csv')
samples = []
for f in main_files:
    df = pd.read_csv(f)
    full_paths = ROOT_DIR + 'hwd/' + df['csv']
    samples.extend(full_paths.tolist())

# Write the list to a file that
with open(ROOT_DIR + 'samples.txt', 'w') as f:
    f.write('\n'.join(samples))

print(f"Extracted {len(samples)} data files")

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

def process_sample(file_path, distance=200) -> np.ndarray | None:
    df = pd.read_csv(file_path, usecols=['X cood.', 'Y cood.', 'Time', 'pen_down'], engine='pyarrow')
    df = df[df['pen_down'] > 0.9]
    df = df.sort_values('Time').reset_index(drop=True)
    df = df.rename(columns={'X cood.': 'x', 'Y cood.': 'y'})

    if len(df) < 2:
        return None

    xy = df[['x', 'y']].to_numpy(dtype=np.float32)
    kept = _filter_by_distance(xy, distance)
    xy = xy[kept]

    if len(xy) < 10:
        return None

    return xy


def _filter_by_distance(xy: np.ndarray, min_dist: float) -> np.ndarray:
    if len(xy) == 0:
        return np.array([], dtype=bool)

    # Compute displacement between every consecutive pair
    deltas = np.diff(xy, axis=0)
    dists  = np.sqrt((deltas ** 2).sum(axis=1))

    # Cumulative distance travelled along the path
    cumdist = np.concatenate([[0.0], np.cumsum(dists)])

    # Keep a point whenever we've travelled >= min_dist since the last kept point
    kept    = [0]
    last_d  = 0.0
    # This loop is over *kept* points (much shorter than N), not all N rows
    thresholds = cumdist
    for i in range(1, len(xy)):
        if thresholds[i] - last_d >= min_dist:
            kept.append(i)
            last_d = thresholds[i]

    return np.array(kept, dtype=np.int64)


def plot_sample(data: np.ndarray) -> None:
    plt.figure(figsize=(8, 1))
    plt.scatter(data[:, 0], data[:, 1], s=0.2, c='blue', alpha=0.8)
    plt.axis('equal')
    plt.axis('off')
    plt.gca().invert_yaxis()
    plt.show()


def process_samples(file_paths: list[str], distance: int = 100, n_workers: int = 4, out_dir: str | None = None) -> list[np.ndarray]:
    if out_dir:
        Path(out_dir).mkdir(exist_ok=True)

    results = {}
    with ProcessPoolExecutor(max_workers=n_workers) as pool:
        futures = {
            pool.submit(_worker, p, distance, out_dir): p
            for p in file_paths
        }
        for fut in tqdm(as_completed(futures), total=len(file_paths)):
            path = futures[fut]
            try:
                results[path] = fut.result()
            except Exception as e:
                print(f"Failed: {path} — {e}")
                results[path] = None

    # Return in original order, filtering out None
    return [results[p] for p in file_paths if results.get(p) is not None]


def _worker(path: str, distance: int, out_dir: str | None):
    """Runs in a subprocess — must be a module-level function (pickle requirement)."""
    arr = process_sample(path, distance=distance)
    if arr is None:
        return None

    # Normalize
    centroid = arr.mean(axis=0)
    arr = arr - centroid
    scale = np.max(np.abs(arr)) + 1e-8
    arr = arr / scale


    if out_dir:
        filename = f"csv_{Path(path).parent.parent.name.split('_')[-1]}_{Path(path).stem.split('_', 1)[1]}"
        np.savez(
            Path(out_dir) / f"{filename}.npz",
            points=arr,
            gt_order=np.arange(len(arr), dtype=np.int64)
        )
        return filename
    return arr

In [ ]:
sample_paths = open(ROOT_DIR + 'samples.txt').read().splitlines()
data = process_samples(sample_paths, out_dir = ROOT_DIR + 'processed/')

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

class HandwritingDataset(Dataset):
    def __init__(self, npz_dir: str):
        self.files = list(Path(npz_dir).glob('*.npz'))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data = np.load(self.files[idx])
        points = data['points']
        N = len(points)

        # Randomly permute the points — this IS the task input
        perm = np.random.permutation(N)
        shuffled_points = points[perm]

        gt_order = np.argsort(perm).astype(np.int64)

        return (
            torch.tensor(shuffled_points, dtype=torch.float32),
            torch.tensor(gt_order, dtype=torch.long),
        )


def collate_fn(batch):
    points_list, order_list = zip(*batch)

    lengths = torch.tensor([len(p) for p in points_list])

    # Pad points with 0.0, gt_order with -1 (so loss can ignore pad positions)
    points_padded = torch.zeros(len(batch), lengths.max(), 2)
    orders_padded = torch.full((len(batch), lengths.max()), fill_value=-1, dtype=torch.long)

    for i, (pts, ord_) in enumerate(zip(points_list, order_list)):
        n = len(pts)
        points_padded[i, :n] = pts
        orders_padded[i, :n] = ord_

    return points_padded, orders_padded, lengths

dataset = HandwritingDataset(ROOT_DIR + 'processed/')

# 80/20 train-val split
val_size   = int(0.2 * len(dataset))
train_size = len(dataset) - val_size
train_set, val_set = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(
    train_set,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_set,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True
)

points, gt_order, lengths = next(iter(train_loader))


print("points shape :", points.shape)
print("gt_order shape:", gt_order.shape)
print("lengths      :", lengths[:8])     # first 8 sequence lengths
print("padding check:", (gt_order[0] == -1).sum().item(), "pad positions in sample 0")

### Model

#### Network

In [3]:
from torch import nn

class PointerNetwork(nn.Module):
    def __init__(self, d_model=256, nhead=4, num_encoder_layers=10,
                 num_decoder_layers=6, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        # Input projection (2D point -> d_model)
        self.input_proj = nn.Sequential(
            nn.Linear(2, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model),
        )

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.d_model, nhead=nhead, dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)

        # Transformer decoder (replaces GRU)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dropout=dropout, batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_decoder_layers)

        # Learnable start-of-sequence token
        self.sos_token = nn.Parameter(torch.randn(d_model))

        # Pointer attention (unchanged, but now takes decoder output instead of GRU state)
        self.attn_W_enc = nn.Linear(d_model, d_model, bias=False)
        self.attn_W_dec = nn.Linear(d_model, d_model, bias=False)
        self.attn_V     = nn.Linear(d_model, 1,       bias=False)

    # ------------------------------------------------------------------
    # forward – teacher forcing (target_order given) or free-running
    # ------------------------------------------------------------------
    def forward(self, points, lengths, target_order=None):
        B, N, _ = points.shape
        device = points.device

        # --- Encoder ---
        pad_mask = torch.arange(N, device=device).unsqueeze(0) >= lengths.unsqueeze(1)
        h = self.input_proj(points)                                    # (B,N,d_model)
        enc = self.encoder(h, src_key_padding_mask=pad_mask)           # (B,N,d_model)
        enc = enc.masked_fill(pad_mask.unsqueeze(-1), 0.0)

        # Precompute encoder projection (used in pointer scores)
        enc_proj = self.attn_W_enc(enc)                                # (B,N,d_model)

        if target_order is not None:
            # ----------------------------------------------------------
            # TEACHER FORCING – vectorized as before
            # ----------------------------------------------------------
            S = target_order.shape[1]               # number of decoding steps

            # Build decoder input sequence: [SOS, point_t0, point_t1, ...]
            # For safety, clamp padding (-1) to 0; these will be masked later
            idx_safe = target_order.clamp(min=0)

            # Gather encoder embeddings of the points that were chosen
            # enc[ B, target_order ] gives the true point embedding
            # We need the sequence shifted: input[t] = point chosen at step t-1
            tgt_seq = torch.zeros(B, S, self.d_model, device=device)
            tgt_seq[:, 0, :] = self.sos_token.unsqueeze(0).expand(B, -1)   # SOS at t=0
            if S > 1:
                prev_points = enc[torch.arange(B, device=device).unsqueeze(1),
                                  idx_safe[:, :-1]]          # (B, S-1, d_model)
                tgt_seq[:, 1:, :] = prev_points

            # Target padding mask: mark positions where the ground truth is padding
            # (SOS position is never masked)
            tgt_pad_mask = torch.cat([
                torch.zeros(B, 1, dtype=torch.bool, device=device),
                target_order[:, :-1] == -1
            ], dim=1)                                          # (B, S)

            # Causal self-attention mask (no peeking into future)
            causal_mask = torch.triu(
                torch.ones(S, S, device=device) * float('-inf'), diagonal=1
            )                                                  # (S, S)

            # Run the Transformer decoder
            dec_output = self.decoder(
                tgt=tgt_seq,
                memory=enc,
                tgt_mask=causal_mask,
                tgt_key_padding_mask=tgt_pad_mask,
                memory_key_padding_mask=pad_mask,
            )                                                  # (B, S, d_model)

            # Pointer scores (same additive attention)
            dec_proj = self.attn_W_dec(dec_output)             # (B, S, d_model)
            scores = self.attn_V(
                torch.tanh(enc_proj.unsqueeze(1) + dec_proj.unsqueeze(2))
            ).squeeze(-1)                                      # (B, S, N)

            # Causal "already-chosen" mask (same logic as before)
            valid_idx = target_order.clone().clamp(min=0)
            one_hot = torch.zeros(B, S, N, device=device)
            one_hot.scatter_(2, valid_idx.unsqueeze(2), 1.0)
            # Zero out padding positions
            valid_mask = (target_order >= 0).unsqueeze(-1).float()
            one_hot = one_hot * valid_mask
            cumulative = torch.cumsum(one_hot, dim=1)
            chosen_mask = torch.cat([
                torch.zeros(B, 1, N, device=device),
                cumulative[:, :-1, :]
            ], dim=1)
            # Also mask encoder padding
            chosen_mask = (chosen_mask + pad_mask.float().unsqueeze(1)) > 0

            return scores.masked_fill(chosen_mask, -1e4)

        else:
            # ----------------------------------------------------------
            # FREE‑RUNNING – autoregressive loop (training with no TF)
            # ----------------------------------------------------------
            chosen = pad_mask.clone()             # tracks already selected points
            tgt_seq = [self.sos_token.unsqueeze(0).unsqueeze(0).expand(B, 1, -1)]
            logits_seq = []

            for step in range(N):
                # Current decoder input sequence (length = step+1)
                tgt = torch.cat(tgt_seq, dim=1)   # (B, step+1, d_model)

                # Causal mask for current length
                cur_len = tgt.size(1)
                causal_mask = torch.triu(
                    torch.ones(cur_len, cur_len, device=device) * float('-inf'),
                    diagonal=1
                )

                dec_output = self.decoder(
                    tgt=tgt,
                    memory=enc,
                    tgt_mask=causal_mask,
                    memory_key_padding_mask=pad_mask,
                )                                  # (B, cur_len, d_model)

                # Use only the last step’s output for prediction
                dec_state = dec_output[:, -1, :]   # (B, d_model)

                dec_proj = self.attn_W_dec(dec_state).unsqueeze(1)
                scores = self.attn_V(
                    torch.tanh(enc_proj + dec_proj)
                ).squeeze(-1)                      # (B, N)

                scores = scores.masked_fill(chosen, -1e4)
                logits_seq.append(scores)

                # Greedy selection for the next input
                idx = scores.argmax(dim=-1)
                chosen = chosen.scatter(1, idx.unsqueeze(1), True)

                # Feed the encoding of the chosen point as next input
                next_point = enc[torch.arange(B, device=device), idx].unsqueeze(1)
                tgt_seq.append(next_point)

            return torch.stack(logits_seq, dim=1)   # (B, N, N)

    # ------------------------------------------------------------------
    # decode – greedy inference (unchanged interface)
    # ------------------------------------------------------------------
    @torch.no_grad()
    def decode(self, points, lengths):
        self.eval()
        B, N, _ = points.shape
        device = points.device

        # Same encoder as above
        pad_mask = torch.arange(N, device=device).unsqueeze(0) >= lengths.unsqueeze(1)
        h = self.input_proj(points)
        enc = self.encoder(h, src_key_padding_mask=pad_mask)
        enc = enc.masked_fill(pad_mask.unsqueeze(-1), 0.0)
        enc_proj = self.attn_W_enc(enc)

        chosen = pad_mask.clone()
        tgt_seq = [self.sos_token.unsqueeze(0).unsqueeze(0).expand(B, 1, -1)]
        max_steps = lengths.max().item()
        output_indices = []

        for step in range(max_steps):
            tgt = torch.cat(tgt_seq, dim=1)
            cur_len = tgt.size(1)
            causal_mask = torch.triu(
                torch.ones(cur_len, cur_len, device=device) * float('-inf'),
                diagonal=1
            )

            dec_output = self.decoder(
                tgt=tgt,
                memory=enc,
                tgt_mask=causal_mask,
                memory_key_padding_mask=pad_mask,
            )
            dec_state = dec_output[:, -1, :]

            dec_proj = self.attn_W_dec(dec_state).unsqueeze(1)
            scores = self.attn_V(
                torch.tanh(enc_proj + dec_proj)
            ).squeeze(-1)
            scores = scores.masked_fill(chosen, -1e4)

            idx = scores.argmax(dim=-1)
            output_indices.append(idx)

            chosen = chosen.scatter(1, idx.unsqueeze(1), True)
            next_point = enc[torch.arange(B, device=device), idx].unsqueeze(1)
            tgt_seq.append(next_point)

        return torch.stack(output_indices, dim=1)

In [5]:
import torch.nn.functional as F

def pointer_loss(logits, gt_order):
    B, S, N    = logits.shape
    target     = gt_order[:, :S]
    logits_flat = logits.reshape(B * S, N)
    target_flat = target.reshape(B * S)
    return F.cross_entropy(logits_flat, target_flat, ignore_index=-1)

In [ ]:
import torch
import numpy as np

import torch.nn.functional as F

def pointer_loss(logits, gt_order):
    B, S, N    = logits.shape
    target     = gt_order[:, :S]
    logits_flat = logits.reshape(B * S, N)
    target_flat = target.reshape(B * S)
    return F.cross_entropy(logits_flat, target_flat, ignore_index=-1)

def kendall_tau(pred_order, gt_order):
    """
    Kendall tau-b correlation for two sequences.
    Works on NumPy arrays.
    1.0 = perfect agreement, -1.0 = completely reversed, 0.0 = no correlation.
    """
    # Remove padding (negative indices)
    valid = gt_order >= 0
    pred = pred_order[valid]
    gt   = gt_order[valid]
    n = len(gt)
    if n < 2:
        return 1.0

    # Use scipy if available, otherwise compute directly
    try:
        from scipy.stats import kendalltau
        tau, _ = kendalltau(pred, gt, variant='b')
        return tau if not np.isnan(tau) else 1.0
    except ImportError:
        pass

    # Manual tau-b calculation
    concordant = 0
    discordant = 0
    ties_pred = 0
    ties_gt   = 0

    for i in range(n - 1):
        for j in range(i + 1, n):
            a = int(pred[j] - pred[i])
            b = int(gt[j] - gt[i])
            if a == 0:
                ties_pred += 1
            if b == 0:
                ties_gt += 1
            if a * b > 0:
                concordant += 1
            elif a * b < 0:
                discordant += 1
            # if either is zero, it's a tie (already counted)

    total = n * (n - 1) / 2.0
    tau = (concordant - discordant) / np.sqrt((total - ties_pred/2) * (total - ties_gt/2))
    return tau


def adjacency_coherence(pred_order, gt_order):
    """
    Fraction of consecutive point pairs in the GT that are **also consecutive**
    (in either direction) in the predicted order.
    A value of 1.0 means the stroke path is perfectly followed, even if the
    starting point differs.
    """
    valid = gt_order >= 0
    pred_valid = pred_order[valid]
    gt_valid   = gt_order[valid]
    n = len(gt_valid)
    if n < 2:
        return 1.0

    # Build a mapping: point index -> position in predicted order
    pos_map = {p.item(): i for i, p in enumerate(pred_valid)}

    # Check each consecutive pair in GT
    coherent = 0
    total_pairs = n - 1
    for i in range(n - 1):
        a = gt_valid[i].item()
        b = gt_valid[i+1].item()
        pos_a = pos_map.get(a, None)
        pos_b = pos_map.get(b, None)
        if pos_a is not None and pos_b is not None and abs(pos_a - pos_b) == 1:
            coherent += 1
    return coherent / total_pairs


def order_length_ratio(pred_order, gt_order):
    """
    Ratio of length (number of points sorted) in ground‑truth.
    This is a sanity check – should be 1.0 if no padding is leaking.
    """
    return (gt_order >= 0).sum().item() / max(1, len(gt_order))

#### Training

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

class HandwritingDataset(Dataset):
    def __init__(self, npz_dir: str):
        self.files = list(Path(npz_dir).glob('*.npz'))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data = np.load(self.files[idx])
        points = data['points']
        N = len(points)

        # Randomly permute the points — this IS the task input
        perm = np.random.permutation(N)
        shuffled_points = points[perm]

        gt_order = np.argsort(perm).astype(np.int64)

        return (
            torch.tensor(shuffled_points, dtype=torch.float32),
            torch.tensor(gt_order, dtype=torch.long),
        )


def collate_fn(batch):
    points_list, order_list = zip(*batch)

    lengths = torch.tensor([len(p) for p in points_list])

    # Pad points with 0.0, gt_order with -1 (so loss can ignore pad positions)
    points_padded = torch.zeros(len(batch), lengths.max(), 2)
    orders_padded = torch.full((len(batch), lengths.max()), fill_value=-1, dtype=torch.long)

    for i, (pts, ord_) in enumerate(zip(points_list, order_list)):
        n = len(pts)
        points_padded[i, :n] = pts
        orders_padded[i, :n] = ord_

    return points_padded, orders_padded, lengths

dataset = HandwritingDataset(ROOT_DIR + 'processed/')

# 80/20 train-val split
val_size   = int(0.2 * len(dataset))
train_size = len(dataset) - val_size
train_set, val_set = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(
    train_set,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_set,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True
)

points, gt_order, lengths = next(iter(train_loader))


print("points shape :", points.shape)
print("gt_order shape:", gt_order.shape)
print("lengths      :", lengths[:8])     # first 8 sequence lengths
print("padding check:", (gt_order[0] == -1).sum().item(), "pad positions in sample 0")

##### Sanity test

In [ ]:
import torch
import gc
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tiny_loader_iter = iter(train_loader)
points_batch, gt_order_batch, lengths_batch = next(tiny_loader_iter)

SMALL_BATCH_SIZE = 1          # number of samples to overfit on
N_ITER = 2000                  # more steps for thorough overfitting
LEARNING_RATE = 3e-4

points   = points_batch[:SMALL_BATCH_SIZE].to(device)
gt_order = gt_order_batch[:SMALL_BATCH_SIZE].to(device)
lengths  = lengths_batch[:SMALL_BATCH_SIZE].to(device)

print("Max sequence length in batch:", lengths.max().item())

del points_batch, gt_order_batch, lengths_batch
torch.cuda.empty_cache()
gc.collect()

torch.backends.cuda.enable_flash_sdp(True)
model_overfit = PointerNetwork(
    d_model=128, nhead=4,
    num_encoder_layers=6,
    num_decoder_layers=6
).to(device)

optimizer_overfit = torch.optim.AdamW(model_overfit.parameters(), lr=LEARNING_RATE)

# -----------------------------------------------------------------
# Helper: compute metrics for the small batch
# -----------------------------------------------------------------
@torch.no_grad()
def evaluate_overfit(model, points, lengths, gt_order):
    model.eval()
    pred_order = model.decode(points, lengths)

    tau_list, coh_list, acc_list = [], [], []
    for i in range(points.shape[0]):
        L = lengths[i].item()
        pred = pred_order[i, :L].cpu().numpy()
        gt   = gt_order[i, :L].cpu().numpy()

        # Call your previously defined metrics (must be in scope)
        tau_list.append(kendall_tau(pred, gt))
        coh_list.append(adjacency_coherence(pred, gt))
        acc_list.append((pred == gt).mean())

    model.train()
    return {
        'kendall_tau': np.mean(tau_list),
        'adj_coherence': np.mean(coh_list),
        'exact_acc': np.mean(acc_list),
    }

# -----------------------------------------------------------------
# Overfit loop
# -----------------------------------------------------------------
model_overfit.train()
print("Overfitting on a tiny batch to test architecture ...\n")
for step in range(N_ITER + 1):
    optimizer_overfit.zero_grad()

    # Always use teacher forcing during overfit test
    logits = model_overfit(points, lengths, target_order=gt_order)
    loss = pointer_loss(logits, gt_order)
    loss.backward()
    optimizer_overfit.step()

    if step % 100 == 0 or step == N_ITER:
        metrics = evaluate_overfit(model_overfit, points, lengths, gt_order)
        print(
            f"Step {step:4d} | loss {loss.item():.4f} | "
            f"adj_coherence {metrics['adj_coherence']:.3f} | "
            f"kendall_tau {metrics['kendall_tau']:.3f} | "
            f"exact_acc {metrics['exact_acc']:.3f}"
        )

print("\nOverfit sanity check done.")

In [10]:
from torch.optim import AdamW
import torch, gc

torch.cuda.empty_cache()
gc.collect()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

torch._dynamo.config.capture_scalar_outputs = True
torch.backends.cuda.enable_flash_sdp(True)
model = PointerNetwork(d_model=128, nhead=4,num_encoder_layers=6,num_decoder_layers=6).to(device)
# model = torch.compile(model, mode="reduce-overhead", dynamic=True)
optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scaler = torch.amp.GradScaler(device)

def get_tf_ratio(epoch, total_epochs, warmup_tf_epochs=10, initial=0.9, final=0.1, decay_fraction=0.2):
    if epoch < warmup_tf_epochs:
        return 1.0

    # Effective epochs after warmup
    effective_epoch = epoch - warmup_tf_epochs
    effective_total = total_epochs - warmup_tf_epochs
    decay_epochs = int(effective_total * decay_fraction)

    if effective_epoch >= decay_epochs:
        return final

    slope = (final - initial) / decay_epochs
    return initial + slope * effective_epoch

def run_epoch(loader, training, teacher_forcing_ratio=1.0):
    model.train() if training else model.eval()
    total_loss = 0
    total_tokens = 0

    with torch.set_grad_enabled(training):
        for points, gt_order, lengths in loader:
            points   = points.to(device)
            gt_order = gt_order.to(device)
            lengths  = lengths.to(device)

            with torch.autocast(device_type='cuda', dtype=torch.float16):
                use_tf = training and (random.random() < teacher_forcing_ratio)
                logits = model(points, lengths, target_order=gt_order if use_tf else None)
                loss   = pointer_loss(logits, gt_order)

            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()

            n_tokens = (gt_order[:, :logits.shape[1]] >= 0).sum().item()
            total_loss += loss.item() * n_tokens
            total_tokens += n_tokens

    return total_loss / total_tokens


@torch.no_grad()
def run_validation(model, loader, device):
    model.eval()
    total_loss_tf = 0.0
    total_tf_tokens = 0

    # New metrics
    tau_sum = 0.0
    coh_sum = 0.0
    count = 0

    for points, gt_order, lengths in loader:
        points   = points.to(device)
        gt_order = gt_order.to(device)
        lengths  = lengths.to(device)

        # Teacher-forced loss (still useful for smoothness)
        logits_tf = model(points, lengths, target_order=gt_order)
        loss_tf   = pointer_loss(logits_tf, gt_order)
        n_tf = (gt_order[:, :logits_tf.shape[1]] >= 0).sum().item()
        total_loss_tf  += loss_tf.item() * n_tf
        total_tf_tokens += n_tf

        # Free‑running greedy decode
        pred_order = model.decode(points, lengths)

        # Per‑sample metrics
        for b in range(points.shape[0]):
            L = lengths[b].item()
            pred = pred_order[b, :L].cpu().numpy()
            gt   = gt_order[b, :L].cpu().numpy()

            tau = kendall_tau(pred, gt)
            coh = adjacency_coherence(pred, gt)

            tau_sum += tau
            coh_sum += coh
            count   += 1

    val_tf_loss = total_loss_tf / total_tf_tokens
    avg_tau = tau_sum / count
    avg_coh = coh_sum / count

    return {
        'loss_tf': val_tf_loss,
        'kendall_tau': avg_tau,
        'adjacency_coherence': avg_coh,
    }

Training on: cuda


In [ ]:
import time
import os
import csv
import torch
from tqdm.notebook import tqdm
from torch.optim.lr_scheduler import OneCycleLR

# ----------------------------------------------------------------------
# Settings
# ----------------------------------------------------------------------
CSV_PATH = ROOT_DIR + 'training_log.csv'
EPOCHS = 100
PATIENCE = 30

# OneCycleLR settings
MAX_LR = 3e-4
PCT_START = 0.1

TEACHER_FORCING_RATIO = 1.0

fieldnames = ['epoch', 'train_loss', 'val_tfloss', 'kendall_tau',
              'adj_coherence', 'best_coherence', 'tf_ratio',
              'epoch_time_sec', 'lr']

# ----------------------------------------------------------------------
# Scheduler
# ----------------------------------------------------------------------
scheduler = OneCycleLR(
    optimizer,
    max_lr=MAX_LR,
    total_steps=EPOCHS * len(train_loader),
    pct_start=PCT_START,
    anneal_strategy='cos'
)

# ----------------------------------------------------------------------
# CSV initialisation
# ----------------------------------------------------------------------
if not os.path.exists(CSV_PATH) or os.path.getsize(CSV_PATH) == 0:
    with open(CSV_PATH, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

# ----------------------------------------------------------------------
# Training loop
# ----------------------------------------------------------------------
best_coherence = -1.0
epochs_without_improvement = 0

pbar = tqdm(range(EPOCHS), desc='Training')
for epoch in pbar:
    t0 = time.time()

    # Training (always teacher‑forced to keep memory low)
    train_loss = run_epoch(train_loader, training=True,
                           teacher_forcing_ratio=TEACHER_FORCING_RATIO)

    # Validation
    val_metrics = run_validation(model, val_loader, device)

    elapsed = time.time() - t0
    epochs_done = epoch + 1
    eta_secs = elapsed * (EPOCHS - epochs_done)
    eta_str = time.strftime('%H:%M:%S', time.gmtime(eta_secs))
    elapsed_str = time.strftime('%H:%M:%S', time.gmtime(elapsed))

    # --- Append one row and close immediately (makes CSV instantly readable) ---
    with open(CSV_PATH, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerow({
            'epoch':            epochs_done,
            'train_loss':       train_loss,
            'val_tfloss':       val_metrics['loss_tf'],
            'kendall_tau':      val_metrics['kendall_tau'],
            'adj_coherence':    val_metrics['adjacency_coherence'],
            'best_coherence':   best_coherence,
            'tf_ratio':         TEACHER_FORCING_RATIO,
            'epoch_time_sec':   elapsed,
            'lr':               optimizer.param_groups[0]['lr'],
        })

    # --- Early stopping & checkpoint ---
    coherence = val_metrics['adjacency_coherence']
    if coherence > best_coherence:
        best_coherence = coherence
        torch.save(model.state_dict(), ROOT_DIR + 'handwriting.pt')
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    pbar.set_postfix_str(
        f"train {train_loss:.4f} | "
        f"tfloss {val_metrics['loss_tf']:.4f} | "
        f"tau {val_metrics['kendall_tau']:.3f} | "
        f"adj {coherence:.3f} | "
        f"best {best_coherence:.3f} | "
        f"step {elapsed_str} | ETA {eta_str}"
    )

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epochs_done} — "
              f"no improvement in adjacency coherence for {PATIENCE} epochs.")
        break

print(f"Training log saved to {CSV_PATH}")

### Inference/Visualization

In [ ]:
import pandas as pd
import plotly.graph_objects as go

CHART_COL = 'val_tfloss' 

df = pd.read_csv(ROOT_DIR + 'training_log.csv')

NAME_MAP = {
    'train_loss': 'Training Loss',
    'val_tfloss': 'Teacher-forced Val Loss',
    'kendall_tau': 'Kendall Tau',
    'adj_coherence': 'Adjoining Coherence',
    'best_coherence': 'Best Coherence',
    'epoch_time_sec': 'Epoch Time (s)',
    'lr': 'Learning Rate'
}

COLOR_MAP = {
    'train_loss': '#636EFA',
    'val_tfloss': '#EF553B',
    'kendall_tau': '#00CC96',
    'adj_coherence': '#AB63FA',
    'best_coherence': '#FFA15A',
    'epoch_time_sec': '#FF6692',
    'lr': '#19D3F3'
}

def hex_to_rgba(hex_color, opacity):
    hex_color = hex_color.lstrip('#')
    lv = len(hex_color)
    rgb = tuple(int(hex_color[i:i + lv // 3], 16) for i in range(0, lv, lv // 3))
    return f'rgba({rgb[0]}, {rgb[1]}, {rgb[2]}, {opacity})'

display_name = NAME_MAP.get(CHART_COL, CHART_COL)
main_color = COLOR_MAP.get(CHART_COL, '#118DFF')
y_values = df[CHART_COL]

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df['epoch'],
        y=y_values,
        mode='lines+markers',
        line=dict(color=main_color, width=3),
        marker=dict(size=6, color=main_color),
        hovertemplate=f"<b>{display_name}</b>: %{{y:.4f}}<extra></extra>"
    )
)

fig.update_layout(
    height=400,
    width=900,
    title={
        'text': display_name,
        'y': 0.9, 'x': 0.5, 'xanchor': 'center'
    },
    template="plotly_white",
    hovermode="x",
    xaxis=dict(
        showticklabels=False, 
        title=None, 
        showgrid=True,
        gridcolor='#f0f0f0'
    ),
    yaxis=dict(
        title=display_name,
        showgrid=True,
        gridcolor='#f0f0f0',
        zeroline=False
    ),
    margin=dict(l=50, r=20, t=60, b=20)
)

fig.show()

In [ ]:
import numpy as np
import cv2
import networkx as nx
from skimage.morphology import skeletonize

def get_ordered_points_from_image(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    skeleton = skeletonize(binary > 0).astype(np.uint8)
    
    nodes = np.argwhere(skeleton > 0)
    G = nx.Graph()
    for r, c in nodes:
        G.add_node((r, c))
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                if dr == 0 and dc == 0:
                    continue
                if (r + dr, c + dc) in G:
                    G.add_edge((r, c), (r + dr, c + dc))

    components = sorted(nx.connected_components(G),
                        key=lambda c: max(n[1] for n in c), reverse=True)

    all_points = []
    for comp in components:
        subgraph = G.subgraph(comp)
        # Find top-right-most node as start for the first stroke
        start_node = min(comp, key=lambda n: n[0] - n[1])

        strokes = []
        visited = set()

        def trace_stroke(start):
            """Follow one continuous strand until dead-end, always preferring rightmost neighbour."""
            stroke = []
            curr = start
            while True:
                stroke.append(curr)
                visited.add(curr)
                neighbours = [n for n in subgraph.neighbors(curr) if n not in visited]
                if not neighbours:
                    break
                # Right-preference: pick the unvisited neighbour with the largest column index
                curr = sorted(neighbours, key=lambda n: n[1], reverse=True)[0]
            return stroke

        # First stroke from the designated top-right start
        strokes.append(trace_stroke(start_node))

        # Continue with any remaining unvisited branches
        unvisited = set(subgraph.nodes) - visited
        while unvisited:
            next_start = None
            for n in unvisited:
                if any(nbr in visited for nbr in subgraph.neighbors(n)):
                    next_start = n
                    break
            if next_start is None:  # disjoint fragment (should rarely happen)
                next_start = min(unvisited, key=lambda n: n[0] - n[1])
            strokes.append(trace_stroke(next_start))
            unvisited = set(subgraph.nodes) - visited

        for stroke in strokes:
            for node in stroke:
                all_points.append([node[1], node[0]])  # swap to (x, y)

    return np.array(all_points), img

# --- Execution ---
image_path = "urdu_input_2.png"  # Change to your filename
true_points, original_img = get_ordered_points_from_image(image_path)
print(f"Extracted {len(true_points)} points from image.")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import KDTree
import cv2

valid_len = len(true_points)
shuffled_idx = np.random.permutation(valid_len)
input_points = true_points[shuffled_idx]

points_tensor = torch.tensor(input_points, dtype=torch.float32).unsqueeze(0).to(device)
length_tensor = torch.tensor([valid_len], dtype=torch.long).to(device)

model.eval()
with torch.no_grad():
    predicted_indices = model.decode(points_tensor, length_tensor)
    predicted_indices = predicted_indices[0].cpu().numpy()[:valid_len]

pred_pts = input_points[predicted_indices]

tree = KDTree(true_points)
_, matched_gt_indices = tree.query(pred_pts)
variance = np.abs(np.arange(valid_len) - matched_gt_indices)

fig, ax = plt.subplots(1, 2, figsize=(16, 5))
ax[0].imshow(original_img, cmap='gray')
ax[0].set_title("Original Uploaded Image")
ax[0].axis('off')

order_color = np.arange(len(pred_pts))
sc = ax[1].scatter(pred_pts[:, 0], -pred_pts[:, 1],
                   c=order_color, cmap='coolwarm_r', s=10, edgecolors='none')
cbar = plt.colorbar(sc, ax=ax[1], label='Step in predicted sequence (start → end)')
ax[1].set_title("Predicted Sequence Order (red=start, blue=end)")
ax[1].set_aspect('equal')
ax[1].axis('off')
plt.tight_layout()
plt.savefig('predicted_order.png', dpi=300, bbox_inches='tight')
plt.show()
print("Image saved as predicted_order.png")

frames = []
step = max(1, valid_len // 50)
x_lims = [true_points[:, 0].min() - 20, true_points[:, 0].max() + 20]
y_lims = [-true_points[:, 1].max() - 20, -true_points[:, 1].min() + 20]

for i in range(2, valid_len + 1, step):
    fig_v, ax_v = plt.subplots(figsize=(6, 2))
    ax_v.scatter(pred_pts[:i, 0], -pred_pts[:i, 1], c='black', s=2)
    ax_v.set_xlim(x_lims); ax_v.set_ylim(y_lims); ax_v.axis('off')
    
    fig_v.canvas.draw()
    frame = np.frombuffer(fig_v.canvas.buffer_rgba(), dtype='uint8')
    frame = frame.reshape(fig_v.canvas.get_width_height()[::-1] + (4,))
    frames.append(cv2.cvtColor(frame, cv2.COLOR_RGBA2BGR))
    plt.close(fig_v)

if frames:
    height, width = frames[0].shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter('output_prediction.mp4', fourcc, 15, (width, height))
    for f in frames:
        out.write(f)
    out.release()
    print("Video saved as output_prediction.mp4")
else:
    print("No frames generated – check valid_len")